<a href="https://colab.research.google.com/github/yumimint/colab/blob/main/picloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
from collections import namedtuple
from typing import BinaryIO, Callable, Union

import numpy as np


class PicError(Exception):
    pass


class PicReadError(Exception):
    pass


class PicFormatError(PicError):
    pass


class BitStream:
    def __init__(self, buf: BinaryIO, mode: str = "r"):
        if isinstance(buf, bytes):
            buf = io.BytesIO(buf)
        self.buf = buf
        self.temp = 0
        self.bits = 8 if mode == "w" else 0

    def read(self, n: Union[int, str]):
        # bitstring.BitStreamのフリをする
        if isinstance(n, str):
            if n == "bool":
                n = 1
            else:
                num = int(n[n.rfind(":") + 1:])
                if n.startswith("bytes"):
                    assert self.bits == 0
                    return self.buf.read(num)
                n = num

        require = n - self.bits

        if require > 0:
            require = (require + 7) // 8
            bin = self.buf.read(require)
            if len(bin) != require:
                raise PicReadError(
                    (
                        f"Requested {require} bytes could not be read."
                        + f" {require - len(bin)} bytes are missing."
                    )
                )
            self.bits += 8 * len(bin)
            self.temp <<= 8 * len(bin)
            self.temp |= int.from_bytes(bin, "big")

        self.bits -= n
        code = self.temp >> self.bits
        self.temp &= (1 << self.bits) - 1
        return code

    def write(self, bits: int, code: int):
        while bits:
            n = min(bits, self.bits)
            bits, code = self._putpart(bits, code, n)

    def _putpart(self, bits, code, n):
        bits -= n
        part = code >> bits
        code &= (1 << bits) - 1

        self.bits -= n
        self.temp |= part << self.bits

        if self.bits == 0:
            self.buf.write(self.temp.to_bytes())
            self.temp = 0
            self.bits = 8

        return bits, code

    def flush(self):
        if self.bits == 8:
            return
        self.write(self.bits, 0)


def encode_length(bs: BitStream, x: int):
    bits = (x + 1).bit_length() - 1  # 2進数での桁数計算
    lead = (1 << bits) - 1
    bs.write(bits, lead ^ 1)
    bs.write(bits, x - lead)


def decode_length(bs: BitStream) -> int:
    bits = 0
    while True:
        bits += 1
        if bs.read("bool") == 0:
            break
    x = bs.read(f"uint:{bits}")
    x += (1 << bits) - 1
    return x


def read_until(bs: BitStream, term: bytes):
    data = b""
    while True:
        byte = bs.read("bytes:1")
        if byte == term:
            break
        data += byte
    return data


def read_header(bs: BitStream):
    magic = bs.read("bytes:3")
    if magic != b"PIC":
        raise PicFormatError("Magic not match.")

    comment = read_until(bs, b"\x1a")
    dummy = read_until(bs, b"\x00")  # 真のコメントの終わり

    reserve = bs.read("uint:8")
    mode = bs.read("uint:4")
    type = bs.read("uint:4")
    bits = bs.read("uint:16")
    width = bs.read("uint:16")
    height = bs.read("uint:16")
    ext = None
    pal = None
    pal_bits = None
    if type == 0:  # X68k
        if bits in [4, 8]:
            pal_bits = 16
    elif type == 1:  # PC-88VA
        pass
    elif type == 2:  # FM-TOWNS
        pass
    elif type == 3:  # MAC
        pass
    elif type == 15:  # 汎用
        ext = {}
        ext["x"] = bs.read("int:16")  # セーブ座標 Ｘ（－１指定でなし）
        ext["y"] = bs.read("int:16")  # セーブ座標 Ｙ（－１指定でなし）
        ext["w"] = bs.read("uint:8")  # 本当の比率  横
        ext["h"] = bs.read("uint:8")  # 本当の比率  縦
        if bits in [4, 8]:
            pal_bits = bs.read("uint:8")  # パレットのビット長 ( 256/16色の時のみ存在）
            ext["pal_bits"] = pal_bits
    else:
        raise PicFormatError(f"Unknow machine type ({hex(type)})")

    if pal_bits:
        pal = []
        for _ in range(1 << bits):
            pal.append(bs.read(f"uint:{pal_bits}"))

    PICHD = namedtuple(
        "PICHD", "comment dummy reserve mode type bits width height ext pal"
    )
    return PICHD(comment, dummy, reserve, mode, type, bits, width, height, ext, pal)


def decode(bs: BitStream, pix: np.ndarray, read_color: Callable):
    flg = np.zeros(pix.shape, dtype=bool)
    iend = pix.size

    i = -1  # 開始座標(-1, 0)
    c = 0  # 色のスタートは0
    # 最初のピクセル色が0でない場合、空の区画(L=0)が現れる

    while i < iend:
        L = decode_length(bs) - 1  # 区画のピクセル数
        n = min(iend - (i + 1), L)
        while n:  # 区画描画ループ
            n -= 1
            i += 1
            if flg.flat[i]:
                # 連鎖点上を通過した時は、現在の色を変更
                c = pix.flat[i]
                continue
            pix.flat[i] = c

        i += 1  # 次の区画へ進む
        if i >= iend:
            # 最終ピクセルまで描ききった
            break

        c = read_color()
        pix.flat[i] = c

        if bs.read("bool"):
            y, x = np.unravel_index(i, pix.shape)

            while True:
                lr = bs.read("uint:2")
                if lr == 0:
                    if not bs.read("bool"):
                        break
                    else:
                        d = 2 if bs.read("bool") else -2
                elif lr == 1:
                    d = -1
                elif lr == 2:
                    d = 0
                elif lr == 3:
                    d = 1
                y, x = y + 1, x + d
                try:
                    pix[y, x] = c
                    flg[y, x] = True
                except IndexError:
                    pass
                    # 壊れデータ対策


def color_reader(bits: int, bs: BitStream) -> Callable:
    fmt = f"uint:{bits}"

    if bits in [12, 15, 16, 24]:
        cache = ColorCache()

        def read_with_cache():
            if bs.read("bool"):
                c = cache.get(bs.read("uint:7"))
            else:
                c = cache.put(bs.read(fmt))
            return c

        return read_with_cache

    def read_direct():
        c = bs.read(fmt)
        return c

    return read_direct


class ColorFormat(dict):
    """カラーコードのビットフィールドを扱うヘルパークラス"""

    class field:
        def __init__(self, mask: int, shift: int, bits: int):
            self.mask = mask
            self.shift = shift
            self.bits = bits
            self.max = (1 << bits) - 1

        def get(self, v):
            return (v & self.mask) >> self.shift

        def put(self, v, inbits=8):
            return ((v >> (inbits - self.bits)) << self.shift) & self.mask

    order: str
    bits: list[int]

    def __init__(self, format: str):
        self.order = format[::2]
        self.bits = list(map(int, format[1::2]))
        assert len(self.bits) == len(self.order)
        shift = sum(self.bits)
        for ch, bits in zip(self.order, self.bits):
            mask = (1 << shift) - 1
            shift -= bits
            mask ^= (1 << shift) - 1
            self[ch] = self.field(mask, shift, bits)


class ColorCache:
    class Node:
        def __init__(self):
            self.color = 0
            self.next = None
            self.prev = None

        def cut(self):
            self.next.prev = self.prev
            self.prev.next = self.next
            # self.next = self
            # self.prev = self
            return self

        def insert(self, place):
            place.prev.next = self
            self.prev = place.prev
            place.prev = self
            self.next = place
            return self

    def __init__(self):
        """色キャッシュの初期化"""
        self.table = [self.Node() for i in range(128)]
        self.color_p = self.table[0]  # 最新色を指す
        for i, p in enumerate(self.table):
            p.color = 0
            p.prev = self.table[(i + 1) & 127]
            p.next = self.table[i - 1]

    def get(self, idx: int):
        """キャッシュから色を取り出しその色が最新になるように更新する"""
        p = self.table[idx]
        if p != self.color_p:
            self.color_p = p.cut().insert(self.color_p)
        return p.color

    def put(self, color):
        """新しい色をキャッシュに登録"""
        self.color_p = self.color_p.prev
        self.color_p.color = color
        return color

    def find(self, color) -> int:
        for i, p in enumerate(self.table):
            if p.color == color:
                self.get(i)
                return i

        return -1


def encode_color(im, format: str, *, order: str = "rgb", dither=False) -> np.ndarray:
    """NumPy画像をPIC画像へエンコードします"""
    colorf = ColorFormat(format)

    bits = sum(colorf.bits)
    channels = np.moveaxis(im, -1, 0)

    rgb = {ch: channels[order.index(ch)] for ch in "rgb"}

    if dither and bits < 24:
        for ch in "rgb":
            rgb[ch] = dither_grayscale(rgb[ch], colorf[ch].bits)

    dtype = np.uint32 if bits > 16 else (np.uint16 if bits > 8 else np.uint8)

    for ch, a in rgb.items():
        rgb[ch] = a.astype(dtype)

    c = (
        colorf["r"].put(rgb["r"])
        | colorf["g"].put(rgb["g"])
        | colorf["b"].put(rgb["b"])
    )

    if bits == 16:
        i = (rgb["r"] & 7) * 0.299
        i += (rgb["g"] & 7) * 0.587
        i += (rgb["b"] & 7) * 0.114
        c |= i > 3.5

    return c


def dither_grayscale(image: np.ndarray, bits: int) -> np.ndarray:
    """
    8bitグレースケール画像を任意のビット数に量子化し、ディザリングを適用する

    Args:
        image: 入力画像 (np.uint8, 2次元配列)
        bits: 目標ビット数 (1〜7)

    Returns:
        np.ndarray: 量子化・ディザリング後の画像 (0-255の範囲)
    """
    if image.dtype != np.uint8:
        raise ValueError("image must be uint8")
    if image.ndim != 2:
        raise ValueError("image must be grayscale (2D)")
    if not (1 <= bits <= 7):
        raise ValueError("bits must be between 1 and 7")

    # コピーしてfloatで処理
    img = image.astype(np.float32)

    # 量子化レベル数
    levels = 2**bits
    step = 255 / (levels - 1)

    h, w = img.shape

    for y in range(h):
        for x in range(w):
            old_pixel = img[y, x]

            # 量子化
            new_pixel = round(old_pixel / step) * step
            img[y, x] = new_pixel

            # 誤差
            error = old_pixel - new_pixel

            # Floyd–Steinberg 拡散
            if x + 1 < w:
                img[y, x + 1] += error * 7 / 16
            if y + 1 < h:
                if x > 0:
                    img[y + 1, x - 1] += error * 3 / 16
                img[y + 1, x] += error * 5 / 16
                if x + 1 < w:
                    img[y + 1, x + 1] += error * 1 / 16

    # 範囲クリップしてuint8に戻す
    return np.clip(img, 0, 255).astype(np.uint8)


color_format_dict = {
    12: "g4r4b4",
    15: "g5r5b5",
    16: "g5r5b5i1",
    24: "g8r8b8",
}


def decode_color(color, format: str, *, order: str = "rgb") -> np.ndarray:
    """PIC画像をNumPy画像へデコードします"""
    colorf = ColorFormat(format)

    rgb = {ch: field.get(color) / field.max for ch, field in colorf.items()}

    if "i" in rgb:
        i = rgb["i"]
        u = 1 / (1 << (max(colorf.bits) + 1))
        u = i * u
        v = 1 - u
        for ch in "rgb":
            rgb[ch] = rgb[ch] * u + rgb[ch] * v

    a = np.dstack([rgb[ch] for ch in order])
    a = (a * 255).astype(np.uint8)
    return a


def apply_pallet(index_image: np.ndarray, pal: list, format="g5r5b5i1", *, order="rgb"):
    """
    インデックス画像にパレットを適用する
    """
    pal_array = np.array(pal, dtype=np.uint32)

    # index_image の各値を pal_array のインデックスとして扱い、新しい配列を作成します
    pix = pal_array[index_image]

    return decode_color(pix, format, order=order)


def decode_pic(buf: BinaryIO, *, order="rgb"):
    """PICファイルをNumPy画像へデコードします"""
    bs = BitStream(buf)
    hd = read_header(bs)
    dtype = np.uint8 if hd.bits <= 8 else (
        np.uint16 if hd.bits <= 16 else np.uint32)
    read_color = color_reader(hd.bits, bs)
    pixel = np.zeros((hd.height, hd.width), dtype=dtype)
    decode(bs, pixel, read_color)

    if hd.pal:
        pal_bits = (hd.ext and hd.ext.get("pal_bits", 16)) or 16
        if pal_bits != 16:
            print(f"unknown pal_bits: {pal_bits}")
        return apply_pallet(pixel, hd.pal, order=order)

    im = decode_color(pixel, color_format_dict[hd.bits], order=order)

    return im


def encode_pic(
    buf: BinaryIO, pix: np.ndarray, bits: int, *, pal=None, comment: str = ""
):
    bs = BitStream(buf, "w")

    write_header(bs, pix, bits, comment=comment)

    cache = ColorCache()

    def write_color_with_cache(color):
        color = int(color)
        i = cache.find(color)
        if i >= 0:
            bs.write(1, 1)
            bs.write(7, i)
        else:
            bs.write(1, 0)
            bs.write(bits, color)
            cache.put(color)

    def write_color_direct(color: int):
        color = int(color)
        bs.write(1, 0)
        bs.write(bits, color)

    if bits in [8, 4]:
        write_color = write_color_direct
    else:
        write_color = write_color_with_cache

    encode(bs, pix, write_color)
    bs.flush()


def write_header(
    bs: BitStream,
    pix: np.ndarray,
    bits: int,
    *,
    comment: str = "",
    mode: int = 0,
    type: int = 0,
):
    hd = b"PIC"
    hd += comment.encode(encoding="ShiftJIS")
    hd += b"\x1a"
    hd += b"\x00"  # 真のコメントの終わり
    hd += b"\x00"  # 予約
    for b in hd:
        bs.write(8, b)
    bs.write(4, mode)  # # bit 4..7 機種毎のモード
    bs.write(4, type)  # # bit 0..3 機種タイプ
    bs.write(16, bits)
    bs.write(16, pix.shape[1])
    bs.write(16, pix.shape[0])


def encode(bs: BitStream, pix: np.ndarray, write_color):
    flg = np.zeros(pix.shape, dtype=np.bool)

    # 変化点にフラグを立てる
    c = 0
    for i, pc in enumerate(pix.flat):
        if c != pc:
            c = pc
            flg.flat[i] = True

    last = -1
    imax = pix.size
    i = 0
    while i < imax:
        if flg.flat[i]:
            # 変化点を見つけたら区間の長さを書き出す
            encode_length(bs, i - last)
            last = i

            # 次の区画へ進む
            write_color(pix.flat[i])
            encode_chain(bs, pix, flg, i)
        i += 1

    encode_length(bs, i - last)


def encode_chain(bs: BitStream, pix: np.ndarray, flg: np.ndarray, i: int):
    c = pix.flat[i]
    starty, startx = np.unravel_index(i, pix.shape)
    height, width = pix.shape

    chain = []

    y, x = starty + 1, startx
    while y < height:
        for dx in [0, -1, 1, -2, 2]:
            ix = x + dx
            if 0 <= ix < width and flg[y, ix] and pix[y, ix] == c:
                chain.append(dx)
                x = ix
                y += 1
                break
        else:
            break

    if not chain:
        bs.write(1, 0)  # 連鎖ナシ
        return

    bs.write(1, 1)  # 連鎖あります
    chaincode = [
        (2, 2),  # 中    10
        (2, 3),  # 右１  11
        (4, 3),  # 右２  00-11
        (4, 2),  # 左２  00-10
        (2, 1),  # 左１  01
    ]
    y, x = starty, startx
    for dx in chain:
        x += dx
        y += 1
        # assert flg[y, x] and pix[y, x] == c
        flg[y, x] = False
        bits, code = chaincode[dx]
        bs.write(bits, code)
    bs.write(3, 0)  # 00-0 連鎖情報終わり


In [ ]:
import requests

req = requests.get(
    "https://github.com/yumimint/c5c0a7724f4f38517ac5ffc86e129f4e/raw/refs/heads/main/sample.pic")

In [ ]:
read_header(BitStream(req.content))

In [ ]:
from PIL import Image

Image.fromarray(decode_pic(req.content))
